In [1]:
import pandas as pd
from datasets import Dataset, DatasetDict
from transformers import AutoTokenizer
from sklearn.model_selection import train_test_split
import os
# import sys
# sys.path.append('autodl-tmp/simpleT5_TRL')
from simplet5_trl import SimpleT5_TRL
# os.environ['TF_ENABLE_ONEDNN_OPTS'] = '0'
# =========================
# 0) 配置
# =========================
CSV_PATH = "data.with_changescribe.csv"
MODEL_NAME = "/root/autodl-tmp/models/Salesforce/codet5p-220m/codet5p-220m_safetensors"

# =========================
# 2) 读 CSV -> Dataset -> split
# =========================
df = pd.read_csv(CSV_PATH)
label2id={'Adaptive':0, 'Corrective':1, 'Perfective':2}
df = df.replace({"labels": label2id})
df
df.dropna(inplace=True)

2026-02-21 19:46:06.420404: I tensorflow/core/util/port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
2026-02-21 19:46:06.474775: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 AVX512F AVX512_VNNI AVX512_BF16 AVX512_FP16 AVX_VNNI AMX_TILE AMX_INT8 AMX_BF16 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
2026-02-21 19:46:07.488922: I tensorflow/core/util/port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
/root/miniconda3/lib

In [2]:
df['changescribe_text'][0]

'RxJava [Change] ChangeScribeStart\nSummarized Code Changes:\nFile: java/rx/Notification.java\n - Rename identifier `hasException` to `hasThrowable` (1 occurrence) in java/rx/Notification.java\n - Modify conditional expression from `hasException()` to `hasThrowable()` in java/rx/Notification.java\n - Modify conditional expression from `hasException() && !getThrowable().equals(notification.getThrowable())` to `hasThrowable()` in java/rx/Notification.java\n - Remove method call `hasException()` in java/rx/Notification.java\n - Add method call `hasThrowable()` in java/rx/Notification.java\nEnd change part'

In [3]:
PROMPT_PREFIX = """You are an experienced software engineer writing a Git commit subject line.

Write a concise commit message that summarizes the main intent of the following changes.

Constraints:
- One sentence only.
- Use imperative mood.
- Focus on the main intent (e.g., rename, fix, add, remove, refactor).
- Do not include file paths.
- Do not include explanations.
- Do not include quotes or markdown.

Changes:
"""

PROMPT_SUFFIX = "\n\nCommit message:"
import re

def clean_changescribe(s: str) -> str:
    s = re.sub(r"ChangeScribeStart", "", s)
    s = re.sub(r"Summarized Code Changes:", "", s)
    s = re.sub(r"End change part", "", s)
    return s.strip()

# df["clean_changescribe"] = df["changescribe_text"].fillna("").map(clean_changescribe)

# df["gen_prompt"] = df["clean_changescribe"].map(
#     lambda s: PROMPT_PREFIX + s + PROMPT_SUFFIX
# )


df["gen_prompt"] = df["changescribe_text"].fillna("").map(
    lambda s: PROMPT_PREFIX + s.strip() + PROMPT_SUFFIX
)


In [4]:
df['gen_prompt'][0]

'You are an experienced software engineer writing a Git commit subject line.\n\nWrite a concise commit message that summarizes the main intent of the following changes.\n\nConstraints:\n- One sentence only.\n- Use imperative mood.\n- Focus on the main intent (e.g., rename, fix, add, remove, refactor).\n- Do not include file paths.\n- Do not include explanations.\n- Do not include quotes or markdown.\n\nChanges:\nRxJava [Change] \n\nFile: java/rx/Notification.java\n - Rename identifier `hasException` to `hasThrowable` (1 occurrence) in java/rx/Notification.java\n - Modify conditional expression from `hasException()` to `hasThrowable()` in java/rx/Notification.java\n - Modify conditional expression from `hasException() && !getThrowable().equals(notification.getThrowable())` to `hasThrowable()` in java/rx/Notification.java\n - Remove method call `hasException()` in java/rx/Notification.java\n - Add method call `hasThrowable()` in java/rx/Notification.java\n\nCommit message:'

In [5]:
df['msgs'][0]

'Change hasException to hasThrowable--'

In [6]:
# df[df['msgs'].str.contains('trunk')]['msgs'][63]

In [7]:
df = df.rename(columns={'gt_clean':'target_text','gen_prompt':'source_text'})
df

,user,repo,commit,labels,msgs,diffs,feature,target_text,changescribe_text,core_diff,clean_changescribe,source_text
0,ponsonio,RxJava,0531b8bff5c14d9504beefb4ad47f473e3a22932,2,Change hasException to hasThrowable--,diff --git a/rxjava-core/src/main/java/rx/Noti...,"[1, 0, 0, 4, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ...",Change hasException to hasThrowable,RxJava [Change] ChangeScribeStart\nSummarized ...,Summarized Code Changes:\nFile: java/rx/Notifi...,RxJava [Change] \n\nFile: java/rx/Notification...,You are an experienced software engineer writi...
1,ponsonio,RxJava,0950c46beda335819928585f1262dfe1dca78a0b,0,Trying to extend the Scheduler interface accor...,diff --git a/rxjava-core/src/main/java/rx/Sche...,"[2, 44, 0, 0, 30, 0, 0, 1, 18, 0, 0, 0, 0, 0, ...",Trying to extend the Scheduler interface accor...,RxJava [Change] ChangeScribeStart\nSummarized ...,Summarized Code Changes:\nFile: java/rx/Schedu...,RxJava [Change] \n\nFile: java/rx/Scheduler.ja...,You are an experienced software engineer writi...
2,ponsonio,RxJava,0f92fdd8e6422d5b79c610a7fd8409d222315a49,0,RunAsync method for outputting multiple values--,diff --git a/rxjava-contrib/rxjava-async-util/...,"[2, 53, 0, 0, 42, 0, 0, 1, 45, 1, 0, 0, 0, 0, ...",RunAsync method for outputting multiple values,RxJava [Change] ChangeScribeStart\nSummarized ...,Summarized Code Changes:\nFile: util/async/Asy...,RxJava [Change] \n\nFile: util/async/Async.jav...,You are an experienced software engineer writi...
3,ponsonio,RxJava,100f571c9a2835d5a30a55374b9be74c147e031f,1,forEach with Action1 but not Observer--I re-re...,diff --git a/language-adaptors/rxjava-groovy/s...,"[1, 5, 122, 9, 10, 9, 4, 1, 5, 18, 2, 0, 0, 0,...",forEach with Action1 but not Observer I re-rea...,RxJava [Change] ChangeScribeStart\nSummarized ...,Summarized Code Changes:\nFile: lang/groovy/Ob...,RxJava [Change] \n\nFile: lang/groovy/Observab...,You are an experienced software engineer writi...
4,ponsonio,RxJava,191f023cf5253ea90647bc091dcaf55ccdce81cc,1,1.x: Fix Completable swallows- OnErrorNotImple...,diff --git a/src/main/java/rx/Completable.java...,"[1, 1, 0, 0, 0, 0, 0, 1, 21, 0, 0, 0, 0, 0, 0,...",1.x: Completable swallows OnErrorNotImplemente...,RxJava [Change] ChangeScribeStart\nSummarized ...,Summarized Code Changes:\nFile: java/rx/Comple...,RxJava [Change] \n\nFile: java/rx/Completable....,You are an experienced software engineer writi...
...,...,...,...,...,...,...,...,...,...,...,...,...
1776,jenkinsci,clearcase-plugin,51e9da224f80254476a7dc446bca817b505381d8,2,Use a temporary file to decrease memory consum...,diff --git a/src/main/java/hudson/plugins/clea...,"[2, 12, 0, 4, 1, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0,...",Use a temporary file to decrease memory consum...,clearcase-plugin [Change] ChangeScribeStart\nS...,Summarized Code Changes:\nFile: plugins/clearc...,clearcase-plugin [Change] \n\nFile: plugins/cl...,You are an experienced software engineer writi...
1777,jexp,batch-import,609d6c4b1eea2c33d9fb950fcbb9ba9dc1f80fc3,2,added a more memory efficient structure for st...,diff --git a/src/main/java/org/neo4j/batchimpo...,"[10, 159, 29, 35, 9, 2, 1, 5, 106, 0, 4, 8, 0,...",added a more memory efficient structure for st...,batch-import [Change] ChangeScribeStart\nSumma...,Summarized Code Changes:\nFile: neo4j/batchimp...,batch-import [Change] \n\nFile: neo4j/batchimp...,You are an experienced software engineer writi...
1778,hdiv,hdiv,19b650c78a1c76f4fd90274d7f163f863c0d39e4,2,Memory and performance optimizations,diff --git a/hdiv-config/src/main/java/org/hdi...,"[31, 302, 131, 140, 170, 89, 53, 7, 88, 14, 17...",Memory and performance optimizations,hdiv [Change] ChangeScribeStart\nSummarized Co...,Summarized Code Changes:\nFile: config/xml/Con...,hdiv [Change] \n\nFile: config/xml/ConfigBeanD...,You are an experienced software engineer writi...
1779,casidiablo,persistence,d7bf95159df37a3d338ca267dddd3d26b38ec37c,2,Now it is possible to specify the sqlite open ...,diff --git a/pom.xml b/pom.xml\nindex 394263b...

In [8]:
# df['source_text'][0]

In [9]:
# df['diff_compact'][0]

In [10]:
train, temp_df = train_test_split(df, test_size=0.3, random_state=42)
val, test = train_test_split(temp_df, test_size=0.5, random_state=42)

In [11]:
from simplet5_trl import SimpleT5_TRL
import pandas as pd

train_df = train[["source_text", "target_text"]].copy()
eval_df  = val[["source_text", "target_text"]].copy()

# model.train(train_df=train_df, eval_df=eval_df, max_epochs=5)

# Train
model = SimpleT5_TRL()
model.from_pretrained(MODEL_NAME)
# model.train(train_df=train_df, eval_df=eval_df, max_epochs=3,batch_size=4)

In [12]:
model.train(
  train_df=train_df,
  eval_df=eval_df,
  batch_size=4,           # 但你得按上面“必须改1”让它生效
  max_epochs=8,
  early_stopping_patience_epochs=2,
  load_best_model_at_end=True,
  greater_is_better=True,
)


Map:   0%|          | 0/1022 [00:00<?, ? examples/s]

Map:   0%|          | 0/219 [00:00<?, ? examples/s]

Epoch,Training Loss,Validation Loss,Rouge1,Rouge2,Rougel,Rougelsum
1,3.906000,3.694791,13.630400,3.338300,12.918900,12.918900
2,2.531500,3.594730,15.444100,4.315600,15.003200,15.003200
3,1.366300,3.761534,18.797900,5.543900,17.625100,17.625100
4,1.319400,4.014005,17.685400,5.247000,16.518800,16.518800
5,0.269000,4.251870,17.437400,5.296600,16.371500,16.371500
6,0.120900,4.383897,18.306400,6.131500,17.409700,17.409700
7,0.047800,4.563609,18.088000,6.042100,16.844300,16.844300
8,0.023400,4.626314,18.157900,6.321600,17.211400,17.211400


There were missing keys in the checkpoint model loaded: ['encoder.embed_tokens.weight', 'decoder.embed_tokens.weight', 'lm_head.weight'].


In [13]:
# Predict
model.load_model("/root/autodl-tmp/commit_generative_reinforcement_learning/outputs/checkpoint-2048", use_gpu=True)
print(model.predict(df["source_text"][0]))

['Change hasException to hasThrowable']


In [14]:
df["source_text"][0]

'You are an experienced software engineer writing a Git commit subject line.\n\nWrite a concise commit message that summarizes the main intent of the following changes.\n\nConstraints:\n- One sentence only.\n- Use imperative mood.\n- Focus on the main intent (e.g., rename, fix, add, remove, refactor).\n- Do not include file paths.\n- Do not include explanations.\n- Do not include quotes or markdown.\n\nChanges:\nRxJava [Change] \n\nFile: java/rx/Notification.java\n - Rename identifier `hasException` to `hasThrowable` (1 occurrence) in java/rx/Notification.java\n - Modify conditional expression from `hasException()` to `hasThrowable()` in java/rx/Notification.java\n - Modify conditional expression from `hasException() && !getThrowable().equals(notification.getThrowable())` to `hasThrowable()` in java/rx/Notification.java\n - Remove method call `hasException()` in java/rx/Notification.java\n - Add method call `hasThrowable()` in java/rx/Notification.java\n\nCommit message:'